In [ ]:
import fasttext
import os

path = "/Users/nouroudine/Documents/Challenge_Sorbonne_2025/assembly_instructions_with_registers_1.txt"
# Entraînement de FastText sur le texte des instructions
model = fasttext.train_unsupervised(path, model="skipgram", dim=100, minCount=1)

# Créer le répertoire si nécessaire
os.makedirs("./Assembly_Model", exist_ok=True)

# Sauvegarde du modèle
model.save_model("./Assembly_Model/fasttext_assembly_model.bin")

# Afficher l'embedding d'une instruction
print("Embedding pour 'mov eax' :", model.get_word_vector("mov eax"))

In [ ]:
import networkx as nx
from tqdm import tqdm
import fasttext
import re
import os

# Charger ton modèle FastText personnalisé pour Assembly
path = "./Assembly_Model/fasttext_assembly_model.bin"
model = fasttext.load_model(path)

# Expressions régulières pour extraire les nœuds et les arêtes
EDGE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*->\s*"([a-zA-Z0-9]+)"')
NODE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*\[label\s*=\s*"(.*?)"\s*\]')

# Cache pour éviter de recalculer les embeddings des instructions répétées
embedding_cache = {}

# Expressions régulières pour nettoyer l'assembly
MNEMONIC_CLEANER = re.compile(r'[^a-zA-Z]')
REGISTER_REGEX = re.compile(r'\b(e?[abcd]x|e?[sd]i|e?[bs]p|r[0-9]+[dwb]?|xmm[0-9]+|ymm[0-9]+|zmm[0-9]+)\b', re.IGNORECASE)

# Fonction pour parser un fichier CFG
def parse_cfg_file(cfg_path):
    try:
        with open(cfg_path, "r", encoding="utf-8", errors="ignore") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception:
        return []

# Fonction pour nettoyer et extraire l'assembly de la chaîne donnée
def clean_assembly_instruction(label):
    parts = label.split(":", 2)
    instruction = parts[2].strip() if len(parts) >= 3 else parts[-1].strip()
    assembly_instructions = ""  # Initialisation de la variable
    if instruction:
        mnemonic = MNEMONIC_CLEANER.sub('', instruction.split()[0]) if instruction else ""
        # Extraction des registres
        registers = REGISTER_REGEX.findall(instruction)
        # Combiner le mnémonique et les registres
        if mnemonic and registers:
            assembly_instructions = f"{mnemonic} {' '.join(registers)}"
        elif mnemonic:
            assembly_instructions = mnemonic
    return assembly_instructions

# Fonction pour extraire les embeddings FastText
def get_fasttext_embedding(label):
    if label in embedding_cache:
        return embedding_cache[label]
    # Utilisation du modèle personnalisé pour générer les embeddings
    embedding = model.get_sentence_vector(label)
    embedding_cache[label] = embedding
    return embedding

# Fonction pour extraire les features du CFG
def extract_features(name, directory):
    cfg_path = os.path.join(directory, f"{name}.json")

    if os.path.exists(cfg_path):
        try:
            G = nx.DiGraph()
            file = parse_cfg_file(cfg_path)

            node_labels = {}
            labels = []
            embeddings = []

            for line in file:
                if '->' in line:  # Extraction des arêtes
                    edge_match = EDGE_REGEX.match(line)
                    if edge_match:
                        src, dst = edge_match.groups()
                        G.add_edge(src, dst)

                if "[" in line and "]" in line:  # Extraction des nœuds
                    node_match = NODE_REGEX.match(line)
                    if node_match:
                        node, label = node_match.groups()
                        labels.append(label)
                        # Extraire et nettoyer l'instruction d'assembly
                        assembly_instruction = clean_assembly_instruction(label)
                        node_labels[node] = assembly_instruction
                        embeddings.append(assembly_instruction)
                    else:
                        assembly_instruction = ""

            # Extraire les embeddings avec tqdm
            for node, assembly in tqdm(node_labels.items(), desc="Processing FastText embeddings"):
                G.add_node(node, assembly_vector=get_fasttext_embedding(assembly))

        except Exception as e:
            print(f"Error processing {cfg_path}: {e}")
            return None

        return G, label
    else:
        print(f"File {cfg_path} does not exist.")
        return None

# Test
directory = "/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/cfg_files"


In [ ]:
G = extract_features("198923e65586cdb01efd9512728879388b1169e5236f4dae1553904b21b84faa", directory)[0]

if G:
    print(f"Nombre de nœuds : {G.number_of_nodes()}")
    print(f"Nombre d'arêtes : {G.number_of_edges()}")
    first_node = list(G.nodes())[800]
    print(f"Embedding du premier nœud : {G.nodes[first_node].get('label', '')}")  # Affiche les 10 premiers éléments

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def plot_embeddings(embeddings, method="pca"):
    embeddings = np.array(embeddings)

    if method == "pca":
        reducer = PCA(n_components=2)
        title = "PCA Visualization"
    elif method == "tsne":
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
        title = "t-SNE Visualization"
    else:
        raise ValueError("Method must be 'pca' or 'tsne'")

    reduced_embeddings = reducer.fit_transform(embeddings)

    plt.figure(figsize=(10, 6))
    plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], alpha=0.5)
    plt.title(title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.grid()
    plt.show()

# Extraire les embeddings des nœuds
G, _ = extract_features("198923e65586cdb01efd9512728879388b1169e5236f4dae1553904b21b84faa", directory)

if G:
    embeddings = [data["assembly_vector"] for _, data in G.nodes(data=True)]
    
    print("🔹 Visualisation avec PCA...")
    plot_embeddings(embeddings, method="pca")

    print("🔹 Visualisation avec t-SNE...")
    plot_embeddings(embeddings, method="tsne")


In [ ]:
import torch
from torch_geometric.data import Data
import networkx as nx

def convert_networkx_to_pyg(graph: nx.DiGraph):
    # Créer un mappage des nœuds vers des indices numériques
    node_mapping = {node: i for i, node in enumerate(graph.nodes())}
    
    # Extraire les informations des nœuds et des arêtes
    node_features = []
    for node in graph.nodes(data=True):
        # Si tu as des attributs, tu peux extraire des vecteurs spécifiques (par exemple, assembly_vector)
        node_features.append(node[1].get('assembly_vector', [0] * 10))  # Par exemple, vecteur de taille 10
    
    # Convertir les nœuds en tensor
    x = torch.tensor(node_features, dtype=torch.float)

    # Extraire les arêtes et mapper les nœuds à leurs indices numériques
    edge_index = []
    for src, dst in graph.edges():
        # Convertir les nœuds source et destination en indices numériques
        src_idx = node_mapping[src]  # Convertir le nœud source en indice numérique
        dst_idx = node_mapping[dst]  # Convertir le nœud destination en indice numérique
        edge_index.append([src_idx, dst_idx])

    # Convertir les arêtes en tensor de type long
    edge_index = torch.tensor(edge_index, dtype=torch.long)

    # Vérifier que l'edge_index a bien deux dimensions
    if edge_index.ndimension() == 1:
        edge_index = edge_index.unsqueeze(0)  # Si c'est un vecteur 1D, faire en sorte que ce soit 2D

    # Transposition et rendre le tensor contigu
    edge_index = edge_index.t().contiguous()

    # Optionnel : Ajouter des attributs pour les arêtes si nécessaire
    # edge_attr = torch.ones(edge_index.shape[1], 1)  # Valeur d'exemple, tu peux la personnaliser

    # Calcul des features globales
    num_nodes = len(graph.nodes())
    num_edges = len(graph.edges())
    avg_degree = sum(dict(graph.degree()).values()) / num_nodes if num_nodes > 0 else 0
    graph_density = num_edges / (num_nodes * (num_nodes - 1)) if num_nodes > 1 else 0
    connected_components = nx.number_weakly_connected_components(graph)
    centrality = nx.degree_centrality(graph)  # Calcul de la centralité des nœuds
    avg_centrality = sum(centrality.values()) / len(centrality) if len(centrality) > 0 else 0

    # Ajouter les features globales comme un vecteur
    global_features = torch.tensor([avg_degree, graph_density, connected_components, avg_centrality], dtype=torch.float)

    # Créer un objet PyG Data avec les features globales ajoutées
    data = Data(x=x, edge_index=edge_index, edge_attr=None, global_features=global_features)

    return data

In [ ]:
'''import pandas as pd
df = pd.read_csv("1000_assembly_feature.csv")
graph_names = df['name'].values
labels = torch.tensor(df.drop(columns=['name']).values, dtype=torch.float)  # Shape [num_graphs, 453]'''

In [ ]:
'''graphs = []
for name in df['name'].values:
    graphs.append(extract_features(name, directory))
    # print(f"graph: {name} saved")'''

In [ ]:
'''dataset = []
for name, graph in zip(df['name'], graphs):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        # Reshape des labels en [1, 453]
        label = torch.tensor(df[df['name'] == name].drop(columns='name').values, dtype=torch.float)
        data.y = label.reshape(1, -1)  # Shape [1, 453]
        dataset.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset, "graphs_pyg.pkl")'''

In [2]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.data import Data, DataLoader 
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention, global_add_pool, global_mean_pool

In [3]:
# Charger les objets Data

dataset = torch.load("graphs_pyg.pkl", weights_only=False)

In [ ]:
dataset = dataset[:632] + dataset[633:]

In [ ]:
df = pd.read_csv("1000_assembly_feature.csv")
graph_names = df['name'].values
labels = torch.tensor(df.drop(columns=['name']).values, dtype=torch.float)  # Shape [num_graphs, 453]

# 2. Initialisation des données
dataset = dataset[:300]  # Votre dataset

train_data = dataset[:int(0.8 * len(dataset))]
test_data = dataset[int(0.8 * len(dataset)):]

train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)

In [ ]:
# Calcul des poids avec lissage et normalisation
label_counts = df.drop(columns='name').sum(axis=0)
total_samples = len(df)
epsilon = 1e-3

pos_weights = torch.tensor((total_samples - label_counts + epsilon) / (label_counts + epsilon), dtype=torch.float)

# Clipping des valeurs extrêmes
pos_weights = torch.clamp(pos_weights, min=1, max=100)

# Normalisation logarithmique
pos_weights = torch.log1p(pos_weights)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch_geometric.nn as pyg_nn

class Graph2VecGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Graph2VecGCN, self).__init__()
        # GCN Layer 1
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        # GCN Layer 2
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        # Dropout Layer
        self.dropout = nn.Dropout(p=0.5)  # 50% de dropout pour éviter le surapprentissage
        # Global Pooling Layer
        self.pool = pyg_nn.global_mean_pool
        # MLP for graph-level output
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim + 4, hidden_dim),  # Ajout des features globales
            nn.ReLU(),
            nn.Dropout(p=0.5),  # Dropout avant la couche finale
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, data):
        x, edge_index, batch, global_features = data.x, data.edge_index, data.batch, data.global_features

        # Vérification de la dimension de x pour éviter l'erreur
        if len(x.shape) > 1:
            x = F.normalize(x, p=2, dim=1)  # Normalisation le long de la dimension des caractéristiques (si x est 2D)
        else:
            x = F.normalize(x, p=2, dim=0)  # Sinon, normalisation le long de la dimension batch (si x est 1D)

        # GCN layers
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)  # Dropout après la première couche
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)  # Dropout après la deuxième couche

            # Global pooling (pour obtenir un embedding fixe du graphe)
        x = self.pool(x, batch)

        # Répéter global_features pour correspondre à la taille du batch
        global_features = global_features.unsqueeze(0).expand(x.size(0), -1)

        # Concatenation des features globales
        x = torch.cat((x, global_features), dim=1)

        # MLP pour la prédiction
        out = self.mlp(x)
        return out

In [ ]:
input_dim = 100  # Exemple : taille du vecteur de caractéristiques par nœud
hidden_dim = 128
output_dim = 453  # Nombre de labels à prédire
model = Graph2VecGCN(input_dim, hidden_dim, output_dim)

# Optimiseur et fonction de perte
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)  # Utilisation des poids pour gérer les classes déséquilibrées

# Entraînement du modèle
def train(model, train_loader):
    model.train()
    total_loss = 0
    for data in tqdm(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Évaluation du modèle
def evaluate(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for data in tqdm(test_loader):
            output = model(data)
            preds = torch.sigmoid(output)  # Activation sigmoïde pour les sorties binaires
            all_preds.append(preds)
            all_labels.append(data.y)
    
    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    # Calcul du F1-score (ou une autre métrique de classification)
    f1 = f1_score(all_labels.cpu().numpy(), all_preds.cpu().numpy() > 0.5, average='macro', zero_division=0)
    return f1

In [ ]:
# Entraînement et évaluation
num_epochs = 5
for epoch in tqdm(range(num_epochs)):
    train_loss = train(model, train_loader)
    f1 = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f} |", f"Test F1-Score: {f1:.4f}" )

# Évaluer le modèle sur le test set
f1 = evaluate(model, test_loader)
print(f"Test F1-Score: {f1:.4f}")

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, TransformerConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm
import pandas as pd
import numpy as np

# =================== Préparation des données ===================
# Charger les features d'assemblage
df = pd.read_csv("1000_assembly_feature.csv")
graph_names = df['name'].values
labels = torch.tensor(df.drop(columns=['name']).values, dtype=torch.float)  # [num_graphs, 453]

# Charger le dataset de graphes (assure-toi que 'graphs_pyg.pkl' contient bien des objets PyG Data)
dataset = torch.load("graphs_pyg.pkl", map_location='cpu')
dataset = dataset[:300]  # On utilise 300 graphes pour cet essai

# Split train/test
train_data = dataset[:int(0.8 * len(dataset))]
test_data = dataset[int(0.8 * len(dataset)):]
print(f"Train graphs: {len(train_data)} | Test graphs: {len(test_data)}")

# DataLoader optimisé (batch size 4, plusieurs workers, pin_memory)
train_loader = DataLoader(train_data, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

# =================== Calcul des pos_weights ===================
label_counts = df.drop(columns='name').sum(axis=0)
total_samples = len(df)
epsilon = 1e-3
pos_weights = (total_samples - label_counts + epsilon) / (label_counts + epsilon)
# On clippe et on lisse pour éviter les extrêmes
pos_weights = torch.tensor(pos_weights.values, dtype=torch.float)
pos_weights = torch.clamp(pos_weights, min=1, max=100)
pos_weights = torch.log1p(pos_weights)
print("Pos weights:", pos_weights)

# =================== Définition du Focal Loss ===================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return focal_loss.mean()

# =================== Modèle GNN amélioré ===================
class BoostedGraphTransformer(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=256, output_dim=453, num_heads=4):
        super(BoostedGraphTransformer, self).__init__()
        # Première couche GCN classique
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        # TransformerConv pour exploiter l'attention multi-tête sur les voisins\n        self.conv2 = TransformerConv(hidden_dim, hidden_dim, heads=num_heads, dropout=0.1)\n        # La sortie de TransformerConv a une dimension hidden_dim * num_heads\n        self.bn2 = nn.BatchNorm1d(hidden_dim * num_heads)\n        \n        # Pooling global\n        self.pool = global_mean_pool\n        \n        # MLP final pour la classification (fusion des features globales: ici 4 dimensions)\n        self.mlp = nn.Sequential(\n            nn.Linear(hidden_dim * num_heads + 4, hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(0.5),\n            nn.Linear(hidden_dim, output_dim)\n        )\n        self.dropout = nn.Dropout(0.5)\n\n    def forward(self, data):\n        x, edge_index, batch, gf = data.x, data.edge_index, data.batch, data.global_features\n        # Normaliser les features de nœuds\n        x = F.normalize(x, p=2, dim=1)\n        \n        x = F.relu(self.bn1(self.conv1(x, edge_index)))\n        x = self.dropout(x)\n        \n        x = self.conv2(x, edge_index)  # [N, hidden_dim * num_heads]\n        x = F.relu(self.bn2(x))\n        x = self.dropout(x)\n        \n        # Pooling global pour obtenir un vecteur par graphe\n        x = self.pool(x, batch)  \n        \n        # On suppose que gf est déjà de forme [batch, 4]\n        x = torch.cat([x, gf], dim=1)\n        out = self.mlp(x)\n        return out

# =================== Configuration ===================
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print("Using device:", device)
model = BoostedGraphTransformer(input_dim=100, hidden_dim=256, output_dim=453, num_heads=4).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = FocalLoss(alpha=0.5, gamma=2.0)

# Scheduler et early stopping
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
best_f1 = 0.0
patience, wait = 5, 0
num_epochs = 50


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch_geometric.data.data.Data was not an allowed global by default. Please use `torch.serialization.add_safe_globals([Data])` or the `torch.serialization.safe_globals([Data])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights

In [ ]:

# =================== Boucle d'entraînement ===================
for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    for data in tqdm(train_loader, desc=f'Epoch {epoch} Training'):
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data.y.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)
    
    # Évaluation sur le test_loader
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for data in tqdm(test_loader, desc=f'Epoch {epoch} Evaluation'):
            data = data.to(device)
            output = model(data)
            preds = torch.sigmoid(output)
            all_preds.append(preds.cpu())
            all_labels.append(data.y.cpu())
    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    f1 = f1_score(all_labels.numpy(), (all_preds.numpy() > 0.5), average='macro', zero_division=0)
    
    print(f"Epoch {epoch}/{num_epochs} - Loss: {avg_loss:.4f}, F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        wait = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"Best F1: {best_f1:.4f}")

In [ ]:
# Sauvegarder le modèle après l'entraînement
torch.save(model.state_dict(), "New_300_data_model.pth")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import f1_score

# Charger le modèle sauvegardé en s'assurant d'utiliser le bon device
model.load_state_dict(torch.load("New_300_data_model.pth"))
model.eval()

def evaluate_all(loader):
    model.eval()
    preds_list, truths_list = [], []
    
    with torch.no_grad():
        for data in loader:
            # Passer les données dans le modèle
            out = model(data)
            
            # Appliquer la fonction sigmoïde pour obtenir des probabilités
            probs = torch.sigmoid(out)
            
            # Binariser les probabilités selon un seuil de 0.5
            binary_preds = (probs > 0.019).cpu().numpy()
            
            # Ajouter les résultats aux listes
            preds_list.append(binary_preds)
            truths_list.append(data.y.cpu().numpy())
    
    # Concaténer les résultats (prédictions et vérités)
    preds = np.vstack(preds_list)
    truths = np.vstack(truths_list)
    
    # Calcul du F1 Macro sur l'ensemble des labels
    f1 = f1_score(truths, preds, average='macro', zero_division=0)
    return f1

# Évaluation sur le jeu de test
f1_test_all = evaluate_all(test_loader)
print(f"Test F1 (tous les labels) : {f1_test_all:.4f}")

In [ ]:
# Charger le dataset des noms et labels existants
df = pd.read_csv("../test_set_metadata_to_predict.csv", sep=";")  # Déjà structuré avec name, label1, ..., label453
df

In [ ]:
test_data = torch.load("graphs_pyg_test_1.pkl", weights_only=False)
#test_data += torch.load("graphs_pyg_test_2.pkl", weights_only=False)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)  # Pas de shuffle pour garder l'ordre

In [ ]:
input_dim = 100  # Exemple : taille du vecteur de caractéristiques par nœud
hidden_dim = 128
output_dim = 453  # Nombre de labels à prédire
model = Graph2VecGCN(input_dim, hidden_dim, output_dim)
model.load_state_dict(torch.load("New_300_data_model.pth"))
model.eval()  # Mettre en mode évaluation

In [ ]:
# Créer une liste vide pour stocker les prédictions
predictions_list = []

# Vérifier que `df` contient bien la bonne structure
assert df.shape[1] == 454, f"Erreur : df a {df.shape[1]} colonnes, attendu 454"

# Faire les prédictions sur les données de test
with torch.no_grad():
    for idx, data in enumerate(tqdm(test_loader)):
        if data.x is None or data.edge_index is None or data.x.size(0) == 0:
            print(f"⚠️ Graphe vide détecté à l'index {idx}, rempli avec des zéros.")
            predictions_list.append([0] * (df.shape[1] - 1))  # 0 pour toutes les colonnes sauf "name"
            continue  # On passe au graphe suivant

        output = model(data)  # Prédiction brute
        preds = torch.sigmoid(output)  # Activation sigmoïde
        preds = (preds > 0.019).int()  # Binarisation

        predictions_list.append(preds.cpu().numpy().flatten())  # Ajouter la prédiction

# Convertir en DataFrame temporaire
predictions_df = pd.DataFrame(predictions_list, columns=df.columns[1:])  # Exclut la colonne "name"

# Remplacer les colonnes de prédictions dans `df`
df.iloc[:500, 1:] = predictions_df.iloc[:, 0:]  # On conserve les noms et on remplace uniquement les labels

print("✅ Prédictions enregistrées avec la bonne correspondance des noms.")

In [ ]:
df

In [ ]:
df.to_csv("predictions_test_graphs_100.csv", index=False)

In [ ]:
df.to_excel("predictions_test_GNN.xlsx", index=False)